# C09_E01 — PID con saturación y antiwindup

**Capítulo:** 9 — Control PID en la práctica industrial  
**Identificador:** `C09_E01`  
**Objetivo pedagógico:** Implementar PID industrial básico con antiwindup back-calculation.  
**Librerías:** numpy, matplotlib

## Ejemplo industrial

PID en lazo de temperatura con válvula limitada al 75% de carrera.

---

> **Aviso de uso responsable.** Este notebook es didáctico. No es código de producción. Cualquier implementación real requiere validación independiente. Ver `docs/politica_uso_responsable.md`.

## 1. PID didáctico con saturación y antiwindup back-calculation

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
np.random.seed(0)

def planta(y, u, dt, K=2.0, tau=10.0):
    """Primer orden discreto: dy/dt = (K*u - y)/tau."""
    return y + dt*(K*u - y)/tau

def pid_step(state, e, Kp, Ki, Kd, dt, umin, umax, Tt=1.0):
    """PID con antiwindup back-calculation. Tt es la cte. de descarga."""
    P = Kp*e
    state["I"] += Ki*e*dt
    D = Kd*(e - state["e_prev"])/dt
    u_unsat = P + state["I"] + D
    u_sat = np.clip(u_unsat, umin, umax)
    state["I"] += (u_sat - u_unsat)*dt/Tt
    state["e_prev"] = e
    return u_sat, state

def simular(antiwindup=True, T=100, dt=0.1, SP=5.0, umax=1.5):
    t = np.arange(0, T, dt); ys, us = [], []
    y = 0.0; st = {"I": 0.0, "e_prev": 0.0}
    Tt = 1.0 if antiwindup else 1e9   # Tt enorme = sin antiwindup efectivo
    for _ in t:
        e = SP - y
        u, st = pid_step(st, e, 2.0, 0.5, 0.0, dt, 0.0, umax, Tt=Tt)
        y = planta(y, u, dt); ys.append(y); us.append(u)
    return t, np.array(ys), np.array(us)

## 2. Comparativa con y sin antiwindup

In [2]:
t, y_aw, u_aw = simular(antiwindup=True)
t, y_no, u_no = simular(antiwindup=False)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
ax1.plot(t, y_aw, label="Con antiwindup")
ax1.plot(t, y_no, label="Sin antiwindup", linestyle='--')
ax1.axhline(5.0, ls=':', color='gray', label="Consigna")
ax1.set_ylabel("y"); ax1.legend(); ax1.grid(alpha=0.3)
ax1.set_title("C09_E01 — PID con/sin antiwindup")
ax2.plot(t, u_aw, label="u (con antiwindup)")
ax2.plot(t, u_no, label="u (sin)", linestyle='--')
ax2.axhline(1.5, ls=':', color='red', label="Saturación")
ax2.set_xlabel("t (s)"); ax2.set_ylabel("u")
ax2.legend(); ax2.grid(alpha=0.3)
out = '../../figures/cap09/fig_C09_F01_pid_antiwindup.png'
os.makedirs(os.path.dirname(out), exist_ok=True)
fig.tight_layout(); fig.savefig(out, dpi=120); plt.show()

## 3. Interpretación

Sin antiwindup, el integrador acumula error mientras la salida está saturada en `u = 1.5`; cuando el error cambia de signo, la integral tarda en descargarse y el sistema sufre un sobredisparo prolongado. Con antiwindup back-calculation (factor `Tt = 1`), la integral se ajusta en tiempo real cuando `u_unsat ≠ u_sat`, lo que reduce drásticamente el sobredisparo y el tiempo de establecimiento.